# Notebook 3: Introductory Machine Learning with scikit-learn

**Audience:** students on MSc Advanced Machine Learning, MSc Artificial Intelligence, and MSc Data Science who have completed Notebooks 1 and 2  
**Suggested time:** 90–120 minutes

## Learning outcomes
By the end of this notebook you should be able to:

- explain the distinction between features and a target;
- split data into training and test sets;
- fit a scikit-learn model;
- generate predictions on unseen data;
- calculate basic classification metrics;
- use a preprocessing/model pipeline;
- compare a model with a simple baseline;
- perform a small regression example;
- recognise why data leakage and overfitting matter.

This is an introduction to the **workflow**, not a complete machine-learning course. The datasets are bundled with scikit-learn, so the notebook can run without downloading external data.

## 1. What is machine learning?

In supervised machine learning, we have examples where the desired answer is already known.

- **Features (`X`)** are the input variables used to make a prediction.
- **Target (`y`)** is the value we want to predict.
- **Classification** predicts a category.
- **Regression** predicts a numerical value.

A typical scikit-learn workflow is:

1. load and inspect data;
2. separate features and target;
3. split into training and test data;
4. choose a model;
5. `fit()` the model using training data;
6. `predict()` for unseen test data;
7. evaluate the predictions.

## 2. Imports and versions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## 3. Load the Iris dataset

The Iris dataset is a small, classic classification dataset. Each row describes a flower using four measurements. The target is one of three iris species.

`as_frame=True` asks scikit-learn to return pandas objects.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)

X = iris.data
y = iris.target

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
display(X.head())
display(y.head())

In [ ]:
print("Feature names:")
print(iris.feature_names)

print("\nTarget names:")
print(iris.target_names)

print("\nClass counts:")
print(y.value_counts().sort_index())

It is often convenient to create one DataFrame for exploration.

In [ ]:
iris_df = X.copy()
iris_df["target"] = y
iris_df["species"] = iris_df["target"].map(dict(enumerate(iris.target_names)))

iris_df.head()

### Exercise 1: Inspect the dataset
Using `X`, `y` and `iris_df`:
- print the number of rows and feature columns;
- display summary statistics for the feature columns;
- print the mean petal length for each species.

In [ ]:
# TODO: inspect the dataset.
print("Rows:", len(X))

## 4. A quick visual check

Visualisation is useful for understanding structure before modelling. Here we plot two features and use the target class to choose marker values.

In [ ]:
plt.figure(figsize=(7, 5))

for class_id, species_name in enumerate(iris.target_names):
    class_rows = iris_df["target"] == class_id
    plt.scatter(
        iris_df.loc[class_rows, "sepal length (cm)"],
        iris_df.loc[class_rows, "petal length (cm)"],
        label=species_name,
    )

plt.xlabel("Sepal length (cm)")
plt.ylabel("Petal length (cm)")
plt.title("Iris: two of the four available features")
plt.legend()
plt.show()

A plot can reveal patterns, but a model should still be evaluated on data it did not use during training.

## 5. Train/test split

We hold back part of the dataset as a **test set**.

`random_state=42` makes the split reproducible.  
`stratify=y` keeps the class proportions similar in the training and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Test targets:", y_test.shape)

### Exercise 2: Understand the split
Calculate:
- the percentage of rows in the test set;
- the target-class counts in `y_train`;
- the target-class counts in `y_test`.

In [ ]:
# TODO: inspect the split.
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Start with a baseline

Before using a sophisticated model, it is useful to compare against a very simple rule.

`DummyClassifier(strategy="most_frequent")` always predicts the most frequent training class. A real model should usually improve on a sensible baseline.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_predictions = baseline.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline accuracy: {baseline_accuracy:.3f}")

## 7. First model: k-nearest neighbours

A k-nearest-neighbours (KNN) classifier predicts a class by looking at nearby training examples.

Because KNN depends on distances between features, feature scaling is important. A **pipeline** lets us combine preprocessing and modelling into one object.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

knn_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)

knn_model.fit(X_train, y_train)

knn_predictions = knn_model.predict(X_test)
knn_accuracy = accuracy_score(y_test, knn_predictions)

print(f"KNN accuracy: {knn_accuracy:.3f}")

The key scikit-learn pattern is consistent across many estimators:

```python
model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

This common interface is one reason scikit-learn is easy to experiment with.

### Exercise 3: Inspect predictions
Create a DataFrame containing:
- the true target;
- the predicted target;
- the true species name;
- the predicted species name.

Then display the first 10 rows.

In [ ]:
# TODO: build a prediction DataFrame.
prediction_df = pd.DataFrame({
    "true_target": y_test.to_numpy(),
})
prediction_df.head()

## 8. Evaluation beyond one accuracy number

A **confusion matrix** shows which true classes were predicted as which classes.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, knn_predictions)

display_cm = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names,
)

display_cm.plot()
plt.title("KNN confusion matrix")
plt.show()

A classification report includes precision, recall and F1-score for each class. You will study these metrics in more depth elsewhere; for now, notice that different metrics answer different questions.

In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        knn_predictions,
        target_names=iris.target_names,
        zero_division=0,
    )
)

### Exercise 4: Count errors
Using the arrays `y_test` and `knn_predictions`, calculate:
- the number of correct predictions;
- the number of incorrect predictions;
- the percentage that were incorrect.

In [ ]:
# TODO: calculate correct and incorrect predictions.
print("Number of test examples:", len(y_test))

## 9. Try a different model

A decision tree learns a sequence of feature-based decisions. Unlike KNN, it does not require feature scaling.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42,
)

tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)
tree_accuracy = accuracy_score(y_test, tree_predictions)

print(f"Decision-tree accuracy: {tree_accuracy:.3f}")

Decision trees expose `feature_importances_`. These numbers describe how much the fitted tree used each feature to reduce impurity. They should not automatically be interpreted as causal importance.

In [ ]:
feature_importance = pd.Series(
    tree_model.feature_importances_,
    index=X.columns,
).sort_values(ascending=False)

feature_importance

### Exercise 5: Experiment with K

Fit KNN pipelines using `n_neighbors` values of 1, 3, 5, 7 and 9. Store each test accuracy in a dictionary.

This is a teaching exercise: repeatedly choosing a model based on the test set would leak information from the test set into model selection. Later we will use cross-validation for a cleaner comparison.

In [ ]:
k_values = [1, 3, 5, 7, 9]
accuracy_by_k = {}

# TODO: loop over k_values, fit a pipeline, and store each accuracy.
print(accuracy_by_k)

## 10. Cross-validation

A single train/test split can be affected by exactly which rows happen to fall into each set.

**Cross-validation** repeatedly splits the training data, producing several validation scores. This is commonly used during model selection while leaving the final test set untouched.

In [ ]:
from sklearn.model_selection import cross_val_score

cv_model = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5),
)

cv_scores = cross_val_score(
    cv_model,
    X_train,
    y_train,
    cv=5,
    scoring="accuracy",
)

print("Fold accuracies:", cv_scores)
print(f"Mean CV accuracy: {cv_scores.mean():.3f}")
print(f"Std. dev.: {cv_scores.std():.3f}")

### Exercise 6: Cross-validate several K values
For `k = 1, 3, 5, 7, 9`, calculate the mean 5-fold cross-validation accuracy using **only the training data**.

Store the results in a dictionary called `mean_cv_accuracy_by_k`.

In [ ]:
mean_cv_accuracy_by_k = {}

# TODO: loop through the k values and use cross_val_score.
print(mean_cv_accuracy_by_k)

## 11. Predict probabilities

Some classifiers can produce estimated class probabilities.

For KNN, `predict_proba()` reports the proportion of neighbours belonging to each class.

In [ ]:
probabilities = knn_model.predict_proba(X_test)

probability_df = pd.DataFrame(
    probabilities,
    columns=iris.target_names,
)

probability_df.head()

Each row should sum to approximately 1.

In [ ]:
print(probability_df.sum(axis=1).head())

## 12. Common pitfalls

### Data leakage
Information from the test set must not influence training or preprocessing. Pipelines help because preprocessing is fitted only on the training portion during `fit()` and within each cross-validation fold.

### Overfitting
A model can learn the training data extremely closely but generalise poorly to new data.

### Accuracy is not always enough
For imbalanced or high-stakes tasks, accuracy may hide important behaviour. Other metrics, class balance, error costs and domain context matter.

### Correlation is not causation
Predictive relationships do not by themselves establish causal effects.

## 13. A short regression example

Classification predicts categories. Regression predicts numerical values.

We will create a simple synthetic dataset, fit a linear regression model, and evaluate it using **mean absolute error (MAE)**.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X_reg, y_reg = make_regression(
    n_samples=200,
    n_features=3,
    noise=12.0,
    random_state=42,
)

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.25,
    random_state=42,
)

regression_model = LinearRegression()
regression_model.fit(X_reg_train, y_reg_train)

regression_predictions = regression_model.predict(X_reg_test)
mae = mean_absolute_error(y_reg_test, regression_predictions)

print(f"Mean absolute error: {mae:.2f}")

MAE is the average absolute distance between the prediction and true value, expressed in the same units as the target.

In [ ]:
comparison = pd.DataFrame({
    "actual": y_reg_test[:10],
    "predicted": regression_predictions[:10],
})

comparison["absolute_error"] = (
    comparison["actual"] - comparison["predicted"]
).abs()

comparison

### Exercise 7: Regression check
Calculate the mean absolute error manually from all regression predictions using NumPy:

1. subtract predictions from true values;
2. take absolute values;
3. take the mean.

Confirm that your result matches `mae`.

In [ ]:
# TODO: calculate manual_mae.
manual_mae = None
print("scikit-learn MAE:", mae)
print("manual MAE:", manual_mae)

# Final mini-challenge: Complete a classification workflow

Using the Iris data already loaded:

1. make a new train/test split using `random_state=7`, `test_size=0.20`, and `stratify=y`;
2. create a pipeline containing `StandardScaler()` and `KNeighborsClassifier(n_neighbors=3)`;
3. fit the model;
4. predict the test set;
5. print the accuracy;
6. print a confusion matrix;
7. compare the result with a `DummyClassifier(strategy="most_frequent")` fitted on the same training split.

The goal is not to hunt for the largest score. The goal is to practise the repeatable modelling workflow.

In [ ]:
# TODO: complete the mini-challenge.
print("Use X and y from the Iris dataset.")

# Recap

You have completed a compact end-to-end machine-learning workflow:

- represent data as features (`X`) and a target (`y`);
- split data into training and test sets;
- establish a baseline;
- fit classifiers with `fit()`;
- generate predictions with `predict()`;
- evaluate classification with accuracy and a confusion matrix;
- combine preprocessing and modelling in a pipeline;
- use cross-validation during model comparison;
- fit and evaluate a simple regression model.

The most important habit is to keep a strict separation between **learning from data** and **evaluating on unseen data**.

From here, the MSc can build on these mechanics with statistical reasoning, feature engineering, more model families, model validation, responsible use, reproducibility and deployment.

# Solutions

Solutions to the exercises are collected here so you can attempt each task before checking your answer. These are example solutions: there is often more than one correct way to solve a problem.


## Exercise 1: Inspect the dataset


In [ ]:
print("Rows:", X.shape[0])
print("Feature columns:", X.shape[1])

display(X.describe())

print(
    iris_df.groupby("species")["petal length (cm)"].mean()
)

## Exercise 2: Understand the split


In [ ]:
test_percentage = 100 * len(X_test) / len(X)

print(f"Test percentage: {test_percentage:.1f}%")
print("\nTraining class counts:")
print(y_train.value_counts().sort_index())
print("\nTest class counts:")
print(y_test.value_counts().sort_index())

## Exercise 3: Inspect predictions


In [ ]:
prediction_df = pd.DataFrame({
    "true_target": y_test.to_numpy(),
    "predicted_target": knn_predictions,
})

target_name_map = dict(enumerate(iris.target_names))

prediction_df["true_species"] = prediction_df["true_target"].map(target_name_map)
prediction_df["predicted_species"] = prediction_df["predicted_target"].map(target_name_map)

prediction_df.head(10)

## Exercise 4: Count errors


In [ ]:
correct = np.sum(y_test.to_numpy() == knn_predictions)
incorrect = np.sum(y_test.to_numpy() != knn_predictions)
incorrect_percentage = 100 * incorrect / len(y_test)

print("Correct:", correct)
print("Incorrect:", incorrect)
print(f"Incorrect percentage: {incorrect_percentage:.1f}%")

## Exercise 5: Experiment with K


In [ ]:
k_values = [1, 3, 5, 7, 9]
accuracy_by_k = {}

for k in k_values:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k),
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy_by_k[k] = accuracy_score(y_test, predictions)

print(accuracy_by_k)

## Exercise 6: Cross-validate several K values


In [ ]:
mean_cv_accuracy_by_k = {}

for k in [1, 3, 5, 7, 9]:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k),
    )
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy",
    )
    mean_cv_accuracy_by_k[k] = scores.mean()

print(mean_cv_accuracy_by_k)

## Exercise 7: Regression check


In [ ]:
manual_mae = np.mean(
    np.abs(y_reg_test - regression_predictions)
)

print("scikit-learn MAE:", mae)
print("manual MAE:", manual_mae)

## Final mini-challenge: Complete a classification workflow


In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=7,
    stratify=y,
)

model_2 = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=3),
)
model_2.fit(X_train_2, y_train_2)
predictions_2 = model_2.predict(X_test_2)

model_2_accuracy = accuracy_score(y_test_2, predictions_2)
print(f"KNN accuracy: {model_2_accuracy:.3f}")

cm_2 = confusion_matrix(y_test_2, predictions_2)
ConfusionMatrixDisplay(
    confusion_matrix=cm_2,
    display_labels=iris.target_names,
).plot()
plt.title("Mini-challenge confusion matrix")
plt.show()

baseline_2 = DummyClassifier(strategy="most_frequent")
baseline_2.fit(X_train_2, y_train_2)
baseline_2_predictions = baseline_2.predict(X_test_2)
baseline_2_accuracy = accuracy_score(y_test_2, baseline_2_predictions)

print(f"Baseline accuracy: {baseline_2_accuracy:.3f}")